In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

def load_csv(fname):
    return pd.read_csv(fname)

def summarize_file(df, file_label, name_field=None, top_cmdb_n=20, top_name_n=50, sample_n=10):
    # ensure timestamp exists
    if 'timestamp' in df.columns:
        # convert to datetime with UTC then to Asia/Shanghai
        dt = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        min_ts = dt.min()
        max_ts = dt.max()
        # format compactly
        min_ts_s = min_ts.isoformat() if pd.notnull(min_ts) else None
        max_ts_s = max_ts.isoformat() if pd.notnull(max_ts) else None
    else:
        min_ts_s = None
        max_ts_s = None

    # top cmdb_id values (by frequency)
    if 'cmdb_id' in df.columns:
        top_cmdb = df['cmdb_id'].value_counts().index.tolist()[:top_cmdb_n]
    else:
        top_cmdb = []

    # top name-like field
    if name_field and name_field in df.columns:
        top_names = df[name_field].value_counts().index.tolist()[:top_name_n]
    else:
        # fallback: for error_logs use 'message' if present
        if 'message' in df.columns and name_field is None:
            top_names = df['message'].value_counts().index.tolist()[:top_name_n]
        else:
            top_names = []

    # sample (up to sample_n rows) - keep compact by selecting first sample_n rows
    sample = df.head(sample_n).copy()
    # keep timestamp converted in sample for readability if present
    if 'timestamp' in df.columns:
        sample['timestamp_dt_AsiaShanghai'] = pd.to_datetime(sample['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

    summary = pd.DataFrame([{
        'file': file_label,
        'row_count': int(len(df)),
        'min_timestamp_AsiaShanghai': min_ts_s,
        'max_timestamp_AsiaShanghai': max_ts_s,
        'top_cmdb_ids': top_cmdb,
        'top_names': top_names,
        'sample_rows': sample.shape[0]
    }])
    return summary, sample

# Load files (they are in current working directory)
metric_df = load_csv('metric.csv')
trace_df = load_csv('trace.csv')
log_df = load_csv('log.csv')
error_df = load_csv('error_logs.csv')

# Summarize each
metric_summary, metric_sample = summarize_file(metric_df, 'metric.csv', name_field='kpi_name')
trace_summary, trace_sample = summarize_file(trace_df, 'trace.csv', name_field='trace_name')
log_summary, log_sample = summarize_file(log_df, 'log.csv', name_field='log_name')
error_summary, error_sample = summarize_file(error_df, 'error_logs.csv', name_field='message')

# Display compact summaries and small samples (one summary per file)
metric_summary, metric_sample, trace_summary, trace_sample, log_summary, log_sample, error_summary, error_sample

```
Out[1]:
```
# Summarize execution results in plain English (compact)
def _first_n_repr(lst, n=10):
    if not lst:
        return "[]"
    try:
        if len(lst) > n:
            return str(lst[:n])[:-1] + ", ...]"
        else:
            return str(lst)
    except Exception:
        return str(lst)

metric_row = int(metric_summary.at[0, 'row_count'])
metric_min = metric_summary.at[0, 'min_timestamp_AsiaShanghai']
metric_max = metric_summary.at[0, 'max_timestamp_AsiaShanghai']
metric_cmdb = _first_n_repr(metric_summary.at[0, 'top_cmdb_ids'], 10)
metric_names = _first_n_repr(metric_summary.at[0, 'top_names'], 10)
metric_sample_ct = int(metric_summary.at[0, 'sample_rows'])

trace_row = int(trace_summary.at[0, 'row_count'])
trace_min = trace_summary.at[0, 'min_timestamp_AsiaShanghai']
trace_max = trace_summary.at[0, 'max_timestamp_AsiaShanghai']
trace_cmdb = _first_n_repr(trace_summary.at[0, 'top_cmdb_ids'], 10)
trace_names = _first_n_repr(trace_summary.at[0, 'top_names'], 10)
trace_sample_ct = int(trace_summary.at[0, 'sample_rows'])

log_row = int(log_summary.at[0, 'row_count'])
log_min = log_summary.at[0, 'min_timestamp_AsiaShanghai']
log_max = log_summary.at[0, 'max_timestamp_AsiaShanghai']
log_cmdb = _first_n_repr(log_summary.at[0, 'top_cmdb_ids'], 10)
log_names = _first_n_repr(log_summary.at[0, 'top_names'], 10)
log_sample_ct = int(log_summary.at[0, 'sample_rows'])

error_row = int(error_summary.at[0, 'row_count'])
error_min = error_summary.at[0, 'min_timestamp_AsiaShanghai']
error_max = error_summary.at[0, 'max_timestamp_AsiaShanghai']
error_cmdb = _first_n_repr(error_summary.at[0, 'top_cmdb_ids'], 10)
error_names = _first_n_repr(error_summary.at[0, 'top_names'], 10)
error_sample_ct = int(error_summary.at[0, 'sample_rows'])

summary = (
    "metric.csv: rows={rows}, time_range={min} to {max}, top_cmdb_ids={cmdb}, top_kpi_names={names}, sample_rows={s}.\n"
    "trace.csv: rows={t_rows}, time_range={t_min} to {t_max}, top_cmdb_ids={t_cmdb}, top_trace_names={t_names}, sample_rows={t_s}.\n"
    "log.csv: rows={l_rows}, time_range={l_min} to {l_max}, top_cmdb_ids={l_cmdb}, top_log_names={l_names}, sample_rows={l_s}.\n"
    "error_logs.csv: rows={e_rows}, time_range={e_min} to {e_max}, top_cmdb_ids={e_cmdb}, top_names={e_names}, sample_rows={e_s}."
).format(
    rows=metric_row, min=metric_min, max=metric_max, cmdb=metric_cmdb, names=metric_names, s=metric_sample_ct,
    t_rows=trace_row, t_min=trace_min, t_max=trace_max, t_cmdb=trace_cmdb, t_names=trace_names, t_s=trace_sample_ct,
    l_rows=log_row, l_min=log_min, l_max=log_max, l_cmdb=log_cmdb, l_names=log_names, l_s=log_sample_ct,
    e_rows=error_row, e_min=error_min, e_max=error_max, e_cmdb=error_cmdb, e_names=error_names, e_s=error_sample_ct
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file  row_count min_timestamp_AsiaShanghai max_timestamp_AsiaShanghai                                       top_cmdb_ids                                          top_names  sample_rows
0  metric.csv      29704  2021-03-25T15:30:00+08:00  2021-03-25T16:00:00+08:00  [Mysql01, Mysql02, Tomcat03, Tomcat02, Tomcat0...  [OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAI...           10,     timestamp cmdb_id                                     kpi_name         value timestamp_dt_AsiaShanghai
0  1616657400    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09 2021-03-25 15:30:00+08:00
1  1616657400    IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.303070e+01 2021-03-25 15:30:00+08:00
2  1616657400    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  1.625641e+08 2021-03-25 15:30:00+08:00
3  1616657400    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.846119e+07 2021-03-25 15:30:00+08:00
4  1616657400    IG01    JVM-Operating System_7778_JVM_JVM_CPULoad  6.068000e-01 2021-03-25 15:30:00+08:00
5  1616657400    IG01     JVM-Threads_7778_JVM_ThreadCount_Threads  7.000000e+01 2021-03-25 15:30:00+08:00
6  1616657400    IG01              OSLinux-CPU_CPU-0_SingleCpuUtil  1.630100e+00 2021-03-25 15:30:00+08:00
7  1616657400    IG01              OSLinux-CPU_CPU-0_SingleCpuidle  9.725410e+01 2021-03-25 15:30:00+08:00
8  1616657400    IG01              OSLinux-CPU_CPU-2_SingleCpuUtil  2.194500e+00 2021-03-25 15:30:00+08:00
9  1616657400    IG01              OSLinux-CPU_CPU-2_SingleCpuidle  9.820300e+01 2021-03-25 15:30:00+08:00,         file  row_count min_timestamp_AsiaShanghai max_timestamp_AsiaShanghai                                       top_cmdb_ids                                          top_names  sample_rows
0  trace.csv       7056  2021-03-25T15:30:00+08:00  2021-03-25T16:00:00+08:00  [MG02, MG01, Tomcat03, Tomcat02, dockerB1, IG0...  [trace.self.duration_mean, trace.self.duration...           10,     timestamp cmdb_id                       trace_name        value timestamp_dt_AsiaShanghai
0  1616657400    IG01         trace.self.duration_mean     1.487912 2021-03-25 15:30:00+08:00
1  1616657400    IG01          trace.self.duration_p95     6.853200 2021-03-25 15:30:00+08:00
2  1616657400    IG01             trace.self.row_count  2182.000000 2021-03-25 15:30:00+08:00
3  1616657400    IG01  trace.to_Tomcat01.duration_mean     1.292315 2021-03-25 15:30:00+08:00
4  1616657400    IG01   trace.to_Tomcat01.duration_p95     5.379600 2021-03-25 15:30:00+08:00
5  1616657400    IG01      trace.to_Tomcat01.row_count   273.000000 2021-03-25 15:30:00+08:00
6  1616657400    IG01  trace.to_Tomcat02.duration_mean     1.768621 2021-03-25 15:30:00+08:00
7  1616657400    IG01   trace.to_Tomcat02.duration_p95     8.070750 2021-03-25 15:30:00+08:00
8  1616657400    IG01      trace.to_Tomcat02.row_count   272.000000 2021-03-25 15:30:00+08:00
9  1616657400    IG01  trace.to_Tomcat03.duration_mean     1.437042 2021-03-25 15:30:00+08:00,       file  row_count min_timestamp_AsiaShanghai max_timestamp_AsiaShanghai                                       top_cmdb_ids                                          top_names  sample_rows
0  log.csv        531  2021-03-25T15:30:00+08:00  2021-03-25T16:00:00+08:00  [Tomcat04, Tomcat01, Tomcat03, Tomcat02, apach...  [log.error_count, log.row_count, log.gc_pause_...           10,     timestamp   cmdb_id          log_name   value timestamp_dt_AsiaShanghai
0  1616657400   Mysql01   log.error_count    0.00 2021-03-25 15:30:00+08:00
1  1616657400   Mysql01     log.row_count   40.00 2021-03-25 15:30:00+08:00
2  1616657400   Mysql02   log.error_count    0.00 2021-03-25 15:30:00+08:00
3  1616657400   Mysql02     log.row_count    1.00 2021-03-25 15:30:00+08:00
4  1616657400  Tomcat01   log.error_count    0.00 2021-03-25 15:30:00+08:00
5  1616657400  Tomcat01  log.gc_pause_max    0.08 2021-03-25 15:30:00+08:00
6  1616657400  Tomcat01     log.row_count  558.00 2021-03-25 15:30:00+08:00
7  1616657400  Tomcat02   log.error_count    0.00 2021-03-25 15:30:00+08:00
8  1616657400  Tomcat02  log.gc_pause_max    0.07 2021-03-25 15:30:00+08:00
9  1616657400  Tomcat02     log.row_count  536.00 2021-03-25 15:30:00+08:00,              file  row_count min_timestamp_AsiaShanghai max_timestamp_AsiaShanghai top_cmdb_ids top_names  sample_rows
0  error_logs.csv          0                       None                       None           []        []            0, Empty DataFrame
Columns: [timestamp, cmdb_id, message, timestamp_dt_AsiaShanghai]
Index: [])```
```

In [ ]:
# In[2]:

# Reuse existing metric_df and tz from earlier steps
import pandas as pd

allowed = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
           'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Filter metric rows to allowed components
metric_allowed = metric_df[metric_df['cmdb_id'].isin(allowed)].copy()

# Add localized datetime column (Asia/Shanghai) for proper window filtering and readable min/max
metric_allowed['dt_local'] = pd.to_datetime(metric_allowed['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global aggregations per (cmdb_id, kpi_name) using the entire filtered series
agg_global = (
    metric_allowed
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        p95_value = ('value', lambda s: float(s.quantile(0.95))),
        total_count = ('value', 'size'),
        min_timestamp = ('timestamp', 'min'),
        max_timestamp = ('timestamp', 'max')
    )
)

# Convert min/max timestamps to Asia/Shanghai ISO strings for readability
agg_global['min_timestamp_dt'] = pd.to_datetime(agg_global['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
agg_global['max_timestamp_dt'] = pd.to_datetime(agg_global['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Keep the full aggregated results in kernel variable `agg_global` (as required)

# 2) Define incident window in Asia/Shanghai and filter metric_allowed into that window
window_start = pd.Timestamp('2021-03-25 15:30:00', tz=tz)
window_end   = pd.Timestamp('2021-03-25 16:00:00', tz=tz)

metric_window = metric_allowed[(metric_allowed['dt_local'] >= window_start) & (metric_allowed['dt_local'] <= window_end)].copy()

# 3) For each (cmdb_id, kpi_name) compute window_count
window_counts = (
    metric_window
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(window_count=('value','size'))
)

# 4) Count points in window whose value > global P95
# Merge p95 into window rows, then mark and aggregate
if not metric_window.empty:
    window_with_p95 = metric_window.merge(
        agg_global[['cmdb_id','kpi_name','p95_value']],
        on=['cmdb_id','kpi_name'],
        how='left'
    )
    window_with_p95['above_p95'] = window_with_p95['value'] > window_with_p95['p95_value']
    window_above_counts = (
        window_with_p95
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(window_count_above_p95=('above_p95','sum'))
    )
else:
    # empty placeholder
    window_above_counts = pd.DataFrame(columns=['cmdb_id','kpi_name','window_count_above_p95'])

# 5) Combine results
result_df = (
    agg_global
    .merge(window_counts, on=['cmdb_id','kpi_name'], how='left')
    .merge(window_above_counts, on=['cmdb_id','kpi_name'], how='left')
)

# Fill NaNs for groups with no points in window
result_df['window_count'] = result_df['window_count'].fillna(0).astype(int)
result_df['window_count_above_p95'] = result_df['window_count_above_p95'].fillna(0).astype(int)

# Keep only rows where window_count > 0, select requested columns and sort
result_compact = (
    result_df[result_df['window_count'] > 0]
    [['cmdb_id','kpi_name','p95_value','total_count','window_count','window_count_above_p95']]
    .sort_values(by='window_count_above_p95', ascending=False)
    .reset_index(drop=True)
)

# Limit to top 20 rows for compactness
result_top20 = result_compact.head(20)

# Display the top results (and keep agg_global in memory for later steps)
result_top20

```
Out[2]:
```
# Summarize the computed results in plain English as a single compact string.
# Reuse kernel variables: result_compact, result_top20, agg_global
n_pairs = result_compact.shape[0]
top_rows = result_top20.copy()

lines = [
    f"Summary: Filtered metric.csv to the allowed components and computed global P95 per (cmdb_id, kpi_name).",
    f"{n_pairs} (cmdb_id, kpi_name) pairs had data points in the incident window 2021-03-25 15:30:00 to 16:00:00 (Asia/Shanghai).",
    "Top pairs by points above global P95 (showing up to top 10 from the top-20 table):"
]

for i, r in enumerate(top_rows.head(10).itertuples(index=False), start=1):
    cmdb = r.cmdb_id
    kpi = (r.kpi_name[:120] + '...') if len(r.kpi_name) > 120 else r.kpi_name
    p95 = f"{r.p95_value:.6g}"
    total = int(r.total_count)
    window = int(r.window_count)
    above = int(r.window_count_above_p95)
    lines.append(f"{i}. {cmdb} | {kpi} | p95={p95} | total={total} | window={window} | above_p95={above}")

lines.append("Full aggregated global results are stored in `agg_global`. Per-window results are in `result_compact` and the displayed subset is in `result_top20`.")
summary = "\n".join(lines)

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name     p95_value  total_count  window_count  window_count_above_p95
0   Tomcat01                            OSLinux-CPU_CPU_CPULoad  1.175500e+00           30            30                       2
1    Mysql01             Mysql-MySQL_3306_Handler Read Rnd Next  3.711062e+09           30            30                       2
2    Mysql01                  Mysql-MySQL_3306_Handler Read Key  1.424465e+08           30            30                       2
3    Mysql01                Mysql-MySQL_3306_Handler Read First  6.399458e+06           30            30                       2
4    Mysql01                    Mysql-MySQL_3306_Handler Commit  1.248077e+08           30            30                       2
5    Mysql01                Mysql-MySQL_3306_Created Tmp Tables  9.835196e+05           30            30                       2
6    Mysql01           Mysql-MySQL_3306_Created Tmp Disk Tables  2.781477e+05           30            30                       2
7    Mysql01                        Mysql-MySQL_3306_Com Update  4.658974e+06           30            30                       2
8       MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  9.703040e+08           31            31                       2
9    Mysql01                        Mysql-MySQL_3306_Bytes Sent  8.015692e+10           30            30                       2
10   Mysql01                  Mysql-MySQL_3306_Binlog Cache Use  4.879335e+06           30            30                       2
11   Mysql01                       Mysql-MySQL_3306_Connections  1.439278e+06           30            30                       2
12  Tomcat01                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.858114e+06           30            30                       2
13  Tomcat02                         OSLinux-CPU_CPU_CPUSysTime  1.155111e+00           31            31                       2
14  Tomcat02                         OSLinux-CPU_CPU_CPUCpuUtil  2.878604e+01           22            22                       2
15  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.050000e+01           31            31                       2
16  Tomcat02                        OSLinux-CPU_CPU_CPUUserTime  2.664670e+01           31            31                       2
17   Mysql01                        Mysql-MySQL_3306_Com Select  1.075173e+08           30            30                       2
18   Mysql01        Mysql-MySQL_3306_Innodb dblwr pages written  1.898981e+07           30            30                       2
19   Mysql01               Mysql-MySQL_3306_Innodb data written  6.423848e+11           30            30                       2```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse variables from kernel: result_top20, metric_window, agg_global, tz
# Prepare container for full per-pair window rows (for follow-up, not printed)
window_rows_by_pair = {}

rows = []
for _, row in result_top20.reset_index(drop=True).iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    key = (cmdb, kpi)
    # select window rows for this pair from metric_window (which is already window-filtered and in Asia/Shanghai tz)
    pair_df = metric_window[(metric_window['cmdb_id'] == cmdb) & (metric_window['kpi_name'] == kpi)].copy()
    # store full per-pair window rows for follow-up
    window_rows_by_pair[key] = pair_df
    
    # get p95_value from agg_global
    p95_row = agg_global[(agg_global['cmdb_id'] == cmdb) & (agg_global['kpi_name'] == kpi)]
    p95_value = float(p95_row['p95_value'].values[0]) if not p95_row.empty else float('nan')
    
    # compute max_value_in_window
    if pair_df.empty:
        max_val = np.nan
    else:
        max_val = float(pair_df['value'].max())
    
    # earliest timestamp in window where value > p95
    if pair_df.empty:
        earliest_above = None
    else:
        above_mask = pair_df['value'] > p95_value
        if above_mask.any():
            # dt_local column exists
            ts = pair_df.loc[above_mask, 'dt_local'].min()
            earliest_above = ts.isoformat()
        else:
            earliest_above = None
    
    # max_over_p95_ratio
    if np.isnan(p95_value):
        max_over_ratio = np.nan
    else:
        if p95_value == 0:
            # if p95==0 but max_val>0 -> infinity
            if np.isnan(max_val):
                max_over_ratio = np.nan
            elif max_val > 0:
                max_over_ratio = np.inf
            else:
                # both zero -> zero ratio
                max_over_ratio = 0.0
        else:
            if np.isnan(max_val):
                max_over_ratio = np.nan
            else:
                max_over_ratio = (max_val - p95_value) / p95_value
    
    # longest consecutive minutes above p95
    longest_run = 0
    if not pair_df.empty:
        # sort by timestamp
        pair_sorted = pair_df.sort_values('timestamp').reset_index(drop=True)
        above = pair_sorted['value'] > p95_value
        if above.any():
            # consider only rows where above is True and compute diffs between successive timestamps
            above_df = pair_sorted.loc[above, ['timestamp']].copy().reset_index(drop=True)
            if above_df.shape[0] == 1:
                longest_run = 1
            else:
                # compute diff in seconds between successive above timestamps
                above_df['ts_diff'] = above_df['timestamp'].diff().fillna(0).astype(int)
                # consecutive if ts_diff == 60 (one minute)
                # create group id that increments when ts_diff != 60
                above_df['grp'] = (above_df['ts_diff'] != 60).cumsum()
                run_lengths = above_df.groupby('grp').size()
                longest_run = int(run_lengths.max())
        else:
            longest_run = 0
    else:
        longest_run = 0
    
    rows.append({
        'cmdb_id': cmdb,
        'kpi_name': (kpi[:200]) if isinstance(kpi, str) else kpi,
        'p95_value': p95_value,
        'earliest_timestamp_above_p95': earliest_above,
        'max_value_in_window': max_val,
        'max_over_p95_ratio': max_over_ratio,
        'longest_consecutive_minutes_above_p95': longest_run
    })

# Build result DataFrame
pair_aggregates = pd.DataFrame(rows)

# Sort as requested and limit to top 20
pair_aggregates_sorted = pair_aggregates.sort_values(
    by=['longest_consecutive_minutes_above_p95','max_over_p95_ratio'],
    ascending=[False, False]
).reset_index(drop=True).head(20)

# Display the compact table (keep full per-pair window rows stored in window_rows_by_pair)
pair_aggregates_sorted

```
Out[3]:
```
# Summarize the computed per-pair aggregates in plain English (compact).
# Reuse kernel variables: pair_aggregates_sorted, window_rows_by_pair
lines = []
df = pair_aggregates_sorted.copy()

lines.append(f"Summary of top {len(df)} (cmdb_id, kpi_name) pairs during 2021-03-25 15:30–16:00 (Asia/Shanghai):")
for i, r in enumerate(df.itertuples(index=False), start=1):
    cmdb = r.cmdb_id
    kpi = r.kpi_name
    p95 = r.p95_value
    ear = r.earliest_timestamp_above_p95 or "none"
    maxv = r.max_value_in_window
    ratio = r.max_over_p95_ratio
    run = r.longest_consecutive_minutes_above_p95
    # format numbers compactly
    if pd.isna(p95):
        p95_s = "NaN"
    else:
        p95_s = f"{p95:.6g}"
    if pd.isna(maxv):
        max_s = "NaN"
    else:
        max_s = f"{maxv:.6g}"
    if pd.isna(ratio):
        ratio_s = "NaN"
    elif ratio == float('inf'):
        ratio_s = "inf"
    else:
        ratio_s = f"{ratio:.6g}"
    lines.append(f"{i}. {cmdb} | {kpi[:120]} | p95={p95_s} | earliest_above={ear} | max_in_window={max_s} | max_over_p95_ratio={ratio_s} | longest_consecutive_min={run}")

lines.append("")
lines.append("Key takeaways:")
lines.append("- The top pair by longest consecutive exceedance is Tomcat01 OSLinux_ZABBIX_Host_Uptime (longest run = 2 minutes), but the exceedance magnitude is extremely small (~0.0011%).")
lines.append("- Tomcat01 thread count shows the largest relative jump among the top rows (max_over_p95_ratio ≈ 0.333), with earliest exceedance at 2021-03-25T15:30:00+08:00 and longest run = 1 minute.")
lines.append("- Most MySQL-related KPIs (multiple Mysql01 rows) and other Tomcat/MG KPIs show single-minute exceedances (longest run = 1) with very small ratios over their global P95 (ratios on the order of 1e-4 to 1e-3).")
lines.append("- Full per-pair window rows are stored in the kernel variable `window_rows_by_pair` for any follow-up inspection; the compact summary above is shown for quick triage.")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name     p95_value earliest_timestamp_above_p95  max_value_in_window  max_over_p95_ratio  longest_consecutive_minutes_above_p95
0   Tomcat01                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.858114e+06    2021-03-25T15:59:00+08:00         7.858203e+06            0.000011                                      2
1   Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.050000e+01    2021-03-25T15:30:00+08:00         1.400000e+01            0.333333                                      1
2   Tomcat01                            OSLinux-CPU_CPU_CPULoad  1.175500e+00    2021-03-25T15:44:00+08:00         1.390000e+00            0.182476                                      1
3   Tomcat02                         OSLinux-CPU_CPU_CPUSysTime  1.155111e+00    2021-03-25T15:30:00+08:00         1.274600e+00            0.103443                                      1
4       MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  9.703040e+08    2021-03-25T15:39:00+08:00         1.010217e+09            0.041135                                      1
5   Tomcat02                         OSLinux-CPU_CPU_CPUCpuUtil  2.878604e+01    2021-03-25T15:30:00+08:00         2.975670e+01            0.033720                                      1
6   Tomcat02                        OSLinux-CPU_CPU_CPUUserTime  2.664670e+01    2021-03-25T15:30:00+08:00         2.671690e+01            0.002635                                      1
7    Mysql01                        Mysql-MySQL_3306_Com Update  4.658974e+06    2021-03-25T15:54:00+08:00         4.659517e+06            0.000117                                      1
8    Mysql01                    Mysql-MySQL_3306_Handler Commit  1.248077e+08    2021-03-25T15:54:00+08:00         1.248216e+08            0.000112                                      1
9    Mysql01                  Mysql-MySQL_3306_Binlog Cache Use  4.879335e+06    2021-03-25T15:54:00+08:00         4.879878e+06            0.000111                                      1
10   Mysql01                        Mysql-MySQL_3306_Com Select  1.075173e+08    2021-03-25T15:54:00+08:00         1.075291e+08            0.000110                                      1
11   Mysql01                        Mysql-MySQL_3306_Bytes Sent  8.015692e+10    2021-03-25T15:54:00+08:00         8.016512e+10            0.000102                                      1
12   Mysql01                  Mysql-MySQL_3306_Handler Read Key  1.424465e+08    2021-03-25T15:54:00+08:00         1.424610e+08            0.000102                                      1
13   Mysql01        Mysql-MySQL_3306_Innodb dblwr pages written  1.898981e+07    2021-03-25T15:54:00+08:00         1.899159e+07            0.000093                                      1
14   Mysql01               Mysql-MySQL_3306_Innodb data written  6.423848e+11    2021-03-25T15:54:00+08:00         6.424441e+11            0.000092                                      1
15   Mysql01                Mysql-MySQL_3306_Handler Read First  6.399458e+06    2021-03-25T15:54:00+08:00         6.399959e+06            0.000078                                      1
16   Mysql01             Mysql-MySQL_3306_Handler Read Rnd Next  3.711062e+09    2021-03-25T15:54:00+08:00         3.711334e+09            0.000073                                      1
17   Mysql01                Mysql-MySQL_3306_Created Tmp Tables  9.835196e+05    2021-03-25T15:54:00+08:00         9.835370e+05            0.000018                                      1
18   Mysql01           Mysql-MySQL_3306_Created Tmp Disk Tables  2.781477e+05    2021-03-25T15:54:00+08:00         2.781520e+05            0.000016                                      1
19   Mysql01                       Mysql-MySQL_3306_Connections  1.439278e+06    2021-03-25T15:54:00+08:00         1.439295e+06            0.000012                                      1```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing variables: trace_df, tz, allowed
# Ensure allowed list exists (if not, define)
try:
    allowed
except NameError:
    allowed = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
               'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# 1) Filter trace rows to allowed components (for both global agg and window processing)
trace_allowed = trace_df[trace_df['cmdb_id'].isin(allowed)].copy()

# Add localized datetime column (Asia/Shanghai) for readability and window filtering
trace_allowed['dt_local'] = pd.to_datetime(trace_allowed['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and counts per (cmdb_id, trace_name) across the full trace_allowed
agg_trace_global = (
    trace_allowed
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(
        p95_value = ('value', lambda s: float(s.quantile(0.95))),
        total_count = ('value', 'size'),
        min_timestamp = ('timestamp', 'min'),
        max_timestamp = ('timestamp', 'max')
    )
)

# Convert min/max to localized datetimes for readability
agg_trace_global['min_timestamp_dt'] = pd.to_datetime(agg_trace_global['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
agg_trace_global['max_timestamp_dt'] = pd.to_datetime(agg_trace_global['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Keep agg_trace_global in kernel for follow-up (requirement)

# 2) Filter trace rows to incident window (Asia/Shanghai)
window_start = pd.Timestamp('2021-03-25 15:30:00', tz=tz)
window_end   = pd.Timestamp('2021-03-25 16:00:00', tz=tz)

trace_window = trace_allowed[(trace_allowed['dt_local'] >= window_start) & (trace_allowed['dt_local'] <= window_end)].copy()

# 2) For each (cmdb_id, trace_name) compute total points in window
window_counts = (
    trace_window
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(window_count=('value','size'))
)

# Merge global p95 into window rows to evaluate exceedances
if not trace_window.empty:
    trace_window_with_p95 = trace_window.merge(
        agg_trace_global[['cmdb_id','trace_name','p95_value']],
        on=['cmdb_id','trace_name'],
        how='left'
    )
    trace_window_with_p95['above_p95'] = trace_window_with_p95['value'] > trace_window_with_p95['p95_value']
    # points above p95 per pair
    window_above = (
        trace_window_with_p95
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .agg(window_count_above_p95=('above_p95','sum'))
    )
    # earliest timestamp above p95 per pair
    earliest_above = (
        trace_window_with_p95[trace_window_with_p95['above_p95']]
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .agg(earliest_timestamp_above_p95=('dt_local','min'))
    )
    # convert earliest to ISO string for readability
    if not earliest_above.empty:
        earliest_above['earliest_timestamp_above_p95'] = earliest_above['earliest_timestamp_above_p95'].dt.tz_convert(tz).apply(lambda x: x.isoformat())
        earliest_above = earliest_above[['cmdb_id','trace_name','earliest_timestamp_above_p95']]
else:
    window_above = pd.DataFrame(columns=['cmdb_id','trace_name','window_count_above_p95'])
    earliest_above = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_timestamp_above_p95'])

# 3) Combine results into compact table
trace_result = (
    agg_trace_global
    .merge(window_counts, on=['cmdb_id','trace_name'], how='left')
    .merge(window_above, on=['cmdb_id','trace_name'], how='left')
    .merge(earliest_above, on=['cmdb_id','trace_name'], how='left')
)

# Fill NaNs for pairs with no window points
trace_result['window_count'] = trace_result['window_count'].fillna(0).astype(int)
trace_result['window_count_above_p95'] = trace_result['window_count_above_p95'].fillna(0).astype(int)
trace_result['earliest_timestamp_above_p95'] = trace_result['earliest_timestamp_above_p95'].where(trace_result['earliest_timestamp_above_p95'].notna(), None)

# Keep full aggregated results in kernel variable `agg_trace_global` and per-window rows in `trace_window` (and trace_window_with_p95)
# Filter to rows with window_count>0
trace_result_compact = trace_result[trace_result['window_count'] > 0].copy()

# For sorting by earliest_timestamp_above_p95 ascending, convert to datetime (NaT if None) for sort stability
trace_result_compact['earliest_ts_sort'] = pd.to_datetime(trace_result_compact['earliest_timestamp_above_p95']).dt.tz_localize(None, ambiguous='NaT', nonexistent='NaT')

# Sort as requested: window_count_above_p95 desc, then earliest_timestamp_above_p95 asc (NaT will be last)
trace_result_compact = trace_result_compact.sort_values(
    by=['window_count_above_p95','earliest_ts_sort'],
    ascending=[False, True]
).reset_index(drop=True)

# Select and order requested columns
trace_result_compact = trace_result_compact[[
    'cmdb_id','trace_name','p95_value','total_count','window_count','window_count_above_p95','earliest_timestamp_above_p95'
]]

# Limit to top 20 rows for compactness
trace_result_top20 = trace_result_compact.head(20).copy()

# Store per-window trace rows by pair for follow-up (mapping) - do not print full mappings
trace_window_rows_by_pair = {}
if not trace_window.empty:
    for (cmdb, tname), g in trace_window.groupby(['cmdb_id','trace_name']):
        trace_window_rows_by_pair[(cmdb,tname)] = g.copy()

# Display compact top-20 result
trace_result_top20

```
Out[4]:
```
# Summarize trace analysis results in plain English (compact).
n = trace_result_top20.shape[0]
unique_cmdbs = trace_result_top20['cmdb_id'].unique().tolist()
window_count_breakdown = trace_result_top20['window_count'].value_counts().to_dict()

summary = (
    f"Trace analysis summary:\n"
    f"- Computed global P95 per (cmdb_id, trace_name) for allowed components and examined the incident window 2021-03-25 15:30–16:00 (Asia/Shanghai).\n"
    f"- {n} (cmdb_id, trace_name) pairs had data in the window (top-20 shown). All shown pairs belong to: {unique_cmdbs}.\n"
    f"- For these top pairs, window_count is mostly 31 (some 30); breakdown: {window_count_breakdown}.\n"
    f"- Each listed pair shows window_count_above_p95 = 2, with earliest_timestamp_above_p95 = 2021-03-25T15:30:00+08:00 (i.e., two data points exceeded the global P95, starting at the window start).\n"
    f"- Affected traces are primarily IG01/IG02 self.duration and traces from IG to Tomcat services (duration_mean and duration_p95), indicating short latency spikes at the start of the incident window for the IG components communicating with Tomcat nodes.\n"
    f"- Full global aggregates are stored in `agg_trace_global`. Per-window rows are in `trace_window` and detailed per-pair rows are in `trace_window_rows_by_pair` for follow-up investigation."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name  p95_value  total_count  window_count  window_count_above_p95 earliest_timestamp_above_p95
0     IG01         trace.self.duration_mean   0.701188           31            31                       2    2021-03-25T15:30:00+08:00
1     IG01          trace.self.duration_p95   3.158000           31            31                       2    2021-03-25T15:30:00+08:00
2     IG01  trace.to_Tomcat01.duration_mean   0.680370           31            31                       2    2021-03-25T15:30:00+08:00
3     IG01   trace.to_Tomcat01.duration_p95   3.111300           31            31                       2    2021-03-25T15:30:00+08:00
4     IG01  trace.to_Tomcat02.duration_mean   0.754908           31            31                       2    2021-03-25T15:30:00+08:00
5     IG01   trace.to_Tomcat02.duration_p95   3.230650           31            31                       2    2021-03-25T15:30:00+08:00
6     IG01  trace.to_Tomcat03.duration_mean   0.755842           31            31                       2    2021-03-25T15:30:00+08:00
7     IG01   trace.to_Tomcat03.duration_p95   3.955775           31            31                       2    2021-03-25T15:30:00+08:00
8     IG01  trace.to_Tomcat04.duration_mean   0.686884           30            30                       2    2021-03-25T15:30:00+08:00
9     IG01   trace.to_Tomcat04.duration_p95   3.105997           30            30                       2    2021-03-25T15:30:00+08:00
10    IG02         trace.self.duration_mean   0.687278           31            31                       2    2021-03-25T15:30:00+08:00
11    IG02          trace.self.duration_p95   3.109475           31            31                       2    2021-03-25T15:30:00+08:00
12    IG02  trace.to_Tomcat01.duration_mean   0.630343           31            31                       2    2021-03-25T15:30:00+08:00
13    IG02   trace.to_Tomcat01.duration_p95   3.097750           31            31                       2    2021-03-25T15:30:00+08:00
14    IG02  trace.to_Tomcat02.duration_mean   0.734765           31            31                       2    2021-03-25T15:30:00+08:00
15    IG02   trace.to_Tomcat02.duration_p95   3.230400           31            31                       2    2021-03-25T15:30:00+08:00
16    IG02  trace.to_Tomcat03.duration_mean   0.749677           31            31                       2    2021-03-25T15:30:00+08:00
17    IG02   trace.to_Tomcat03.duration_p95   3.608775           31            31                       2    2021-03-25T15:30:00+08:00
18    IG02  trace.to_Tomcat04.duration_mean   0.730812           31            31                       2    2021-03-25T15:30:00+08:00
19    IG02   trace.to_Tomcat04.duration_p95   3.148600           31            31                       2    2021-03-25T15:30:00+08:00```
```

In [ ]:
# In[5]:

# Reuse kernel variables: trace_window, agg_trace_global, tz
import pandas as pd

# Define targets
target_cmdbs = ['IG01','IG02']
target_traces = [
    'trace.to_Tomcat01.duration_mean',
    'trace.to_Tomcat01.duration_p95',
    'trace.to_Tomcat01.row_count'
]

# Filter per-window trace rows to the targets
trace_targets = trace_window[
    trace_window['cmdb_id'].isin(target_cmdbs) &
    trace_window['trace_name'].isin(target_traces)
].copy()

# Merge global P95 values (computed earlier in agg_trace_global)
trace_targets = trace_targets.merge(
    agg_trace_global[['cmdb_id','trace_name','p95_value']],
    on=['cmdb_id','trace_name'],
    how='left'
)

# Ensure localized timestamp column exists for display
trace_targets['timestamp_dt_AsiaShanghai'] = pd.to_datetime(trace_targets['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Select rows where value > p95_value (handle NaN p95 by treating as False)
mask_above = trace_targets['p95_value'].notna() & (trace_targets['value'] > trace_targets['p95_value'])
matching = trace_targets[mask_above].copy()

# Store full matching rows in kernel variable for follow-up
full_matching_rows = matching.copy()

# Prepare compact table sorted by timestamp ascending, limited to top 50
matching_compact = matching.sort_values('timestamp', ascending=True)[
    ['timestamp','timestamp_dt_AsiaShanghai','cmdb_id','trace_name','value','p95_value']
].reset_index(drop=True).head(50)

# One-line counts per (cmdb_id, trace_name)
counts_summary = matching.groupby(['cmdb_id','trace_name'], as_index=False).agg(exceed_count=('value','size'))

# Display compact results (matching_compact and counts_summary)
matching_compact, counts_summary

```
Out[5]:
```
# Summarize exceeding trace rows for IG01/IG02 -> Tomcat01 in plain English (compact)
# Reuse kernel variables: full_matching_rows, counts_summary, matching_compact
import pandas as pd

# safety: ensure variables exist
try:
    fm = full_matching_rows
    cs = counts_summary
    mc = matching_compact
except NameError:
    fm = pd.DataFrame()
    cs = pd.DataFrame()
    mc = pd.DataFrame()

# compute totals and earliest timestamps per pair
if not fm.empty:
    total_exceeds = int(cs['exceed_count'].sum())
    pairs = []
    earliest_ts = (
        fm.groupby(['cmdb_id','trace_name'], as_index=False)
          .agg(earliest=('timestamp_dt_AsiaShanghai','min'))
    )
    # format per-pair lines
    for _, r in cs.iterrows():
        cmdb = r['cmdb_id']
        tname = r['trace_name']
        cnt = int(r['exceed_count'])
        et = earliest_ts[
            (earliest_ts['cmdb_id']==cmdb) & (earliest_ts['trace_name']==tname)
        ]['earliest']
        et_str = et.iloc[0].isoformat() if not et.empty else "none"
        pairs.append(f"{cmdb} | {tname} : {cnt} rows, earliest={et_str}")
    pairs_text = "; ".join(pairs)
    summary = (
        f"Found {total_exceeds} trace points exceeding their global P95 for IG->Tomcat01 traces.\n"
        f"Counts per (cmdb_id,trace_name): {pairs_text}.\n"
        f"Notable timing: duration_mean and duration_p95 exceedances occurred at the window start (2021-03-25T15:30:00+08:00) for both IG01 and IG02; "
        f"row_count exceedances occurred shortly after (e.g., 15:31 and 15:32) and a few later spikes appear (e.g., 15:46 for IG01 durations).\n"
        f"Compact sample of matching rows is in `matching_compact` and full matching rows are stored in `full_matching_rows` for follow-up."
    )
else:
    summary = "No matching exceeding trace rows found for the requested pairs in the incident window."

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp timestamp_dt_AsiaShanghai cmdb_id                       trace_name       value   p95_value
0   1616657400 2021-03-25 15:30:00+08:00    IG01  trace.to_Tomcat01.duration_mean    1.292315    0.680370
1   1616657400 2021-03-25 15:30:00+08:00    IG01   trace.to_Tomcat01.duration_p95    5.379600    3.111300
2   1616657400 2021-03-25 15:30:00+08:00    IG02  trace.to_Tomcat01.duration_mean    1.471384    0.630343
3   1616657400 2021-03-25 15:30:00+08:00    IG02   trace.to_Tomcat01.duration_p95    7.354200    3.097750
4   1616657460 2021-03-25 15:31:00+08:00    IG01      trace.to_Tomcat01.row_count  327.000000  296.000000
5   1616657460 2021-03-25 15:31:00+08:00    IG02      trace.to_Tomcat01.row_count  313.000000  306.000000
6   1616657520 2021-03-25 15:32:00+08:00    IG02      trace.to_Tomcat01.row_count  316.000000  306.000000
7   1616657640 2021-03-25 15:34:00+08:00    IG02   trace.to_Tomcat01.duration_p95    3.099700    3.097750
8   1616657700 2021-03-25 15:35:00+08:00    IG02  trace.to_Tomcat01.duration_mean    0.633633    0.630343
9   1616657820 2021-03-25 15:37:00+08:00    IG01      trace.to_Tomcat01.row_count  300.000000  296.000000
10  1616658360 2021-03-25 15:46:00+08:00    IG01  trace.to_Tomcat01.duration_mean    0.714089    0.680370
11  1616658360 2021-03-25 15:46:00+08:00    IG01   trace.to_Tomcat01.duration_p95    3.121600    3.111300,   cmdb_id                       trace_name  exceed_count
0    IG01  trace.to_Tomcat01.duration_mean             2
1    IG01   trace.to_Tomcat01.duration_p95             2
2    IG01      trace.to_Tomcat01.row_count             2
3    IG02  trace.to_Tomcat01.duration_mean             2
4    IG02   trace.to_Tomcat01.duration_p95             2
5    IG02      trace.to_Tomcat01.row_count             2)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# ensure tz and source DataFrames exist (reuse from kernel if available)
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df and log_df from the kernel
# 1) Metric KPIs for Tomcat01 across the full metric.csv
tomcat_metric_rows_full = metric_df[metric_df['cmdb_id'] == 'Tomcat01'].copy()

# Add localized datetime column for readability
tomcat_metric_rows_full['timestamp_dt_AsiaShanghai'] = pd.to_datetime(
    tomcat_metric_rows_full['timestamp'], unit='s', utc=True
).dt.tz_convert(tz)

# Aggregate per kpi_name across the full series (global stats)
tomcat_metric_agg = (
    tomcat_metric_rows_full
    .groupby('kpi_name', as_index=False)
    .agg(
        total_count=('value', 'size'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max')
    )
)

# Convert min/max to localized ISO datetimes for compact display
tomcat_metric_agg['min_timestamp_dt'] = pd.to_datetime(tomcat_metric_agg['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
tomcat_metric_agg['max_timestamp_dt'] = pd.to_datetime(tomcat_metric_agg['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Top 50 KPI names by count (compact)
tomcat_metric_agg_top50 = tomcat_metric_agg.sort_values('total_count', ascending=False).head(50)[
    ['kpi_name', 'total_count', 'min_timestamp_dt', 'max_timestamp_dt']
].reset_index(drop=True)

# Up to 10 sample metric rows for Tomcat01 with requested columns
tomcat_metric_sample = tomcat_metric_rows_full[
    ['timestamp', 'timestamp_dt_AsiaShanghai', 'kpi_name', 'value']
].sort_values('timestamp').head(10).reset_index(drop=True)

# 2) Logs for Tomcat01 within the incident window
tomcat_log_rows_full = log_df[log_df['cmdb_id'] == 'Tomcat01'].copy()
tomcat_log_rows_full['timestamp_dt_AsiaShanghai'] = pd.to_datetime(
    tomcat_log_rows_full['timestamp'], unit='s', utc=True
).dt.tz_convert(tz)

# Define window (Asia/Shanghai)
window_start = pd.Timestamp('2021-03-25 15:30:00', tz=tz)
window_end   = pd.Timestamp('2021-03-25 16:00:00', tz=tz)

tomcat_log_window_full = tomcat_log_rows_full[
    (tomcat_log_rows_full['timestamp_dt_AsiaShanghai'] >= window_start) &
    (tomcat_log_rows_full['timestamp_dt_AsiaShanghai'] <= window_end)
].copy()

# Compact view (limit to top 50 rows) with requested columns, sorted by timestamp
tomcat_log_window_top50 = tomcat_log_window_full[
    ['timestamp', 'timestamp_dt_AsiaShanghai', 'log_name', 'value']
].sort_values('timestamp').head(50).reset_index(drop=True)

# Keep full returned tables in kernel variables for follow-up:
# tomcat_metric_agg (full per-kpi aggregation), tomcat_metric_rows_full (all Tomcat01 metric rows),
# tomcat_log_window_full (all Tomcat01 log rows in window)

tomcat_metric_agg_top50, tomcat_metric_sample, tomcat_log_window_top50

```
Out[6]:
```
# Plain-English summary as a single string (compact)
summary = (
    "Tomcat01 compact listing:\n"
    "1) Metrics (metric.csv):\n"
    "- Found up to 50 unique KPI names for Tomcat01 (top-50 returned). Many KPIs have full-series counts of 31 (per-minute series covering 15:30–16:00), while some filesystem/less-frequent KPIs show 15 points.\n"
    "- Global time range per KPI (min/max) is generally within 2021-03-25 15:30:00+08:00 to 2021-03-25 16:00:00+08:00 (some KPIs start at 15:31:00 depending on metric frequency).\n"
    "- Example sample metric rows (earliest 10): CPUCpuUtil=28.197, CPULoad=1.16, CPUSysTime≈0.928, CPUUserTime≈26.36, CPUidleutil≈71.67, Memory CacheMem=3779, FreeMem=118, a process metric=0.3, PROCPPMem=58. (These are representative values from 15:30.)\n"
    "2) Logs (log.csv) in incident window 2021-03-25 15:30–16:00 (Asia/Shanghai):\n"
    "- Returned up to 50 log rows for Tomcat01 within the window (sample shown). Rows span timestamps from 15:30 through at least 15:47 in the sample.\n"
    "- Typical log entries: log.error_count = 0.00 at sampled times; log.gc_pause_max small (e.g., 0.08, 0.03, 0.11); log.row_count values around 558 then lower values later (e.g., 486, 495) in the sample.\n"
    "Stored kernel variables for follow-up: tomcat_metric_agg (per-KPI aggregation), tomcat_metric_rows_full (all Tomcat01 metric rows), and tomcat_log_window_full (all Tomcat01 log rows in window)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  total_count          min_timestamp_dt          max_timestamp_dt
0   Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...           31 2021-03-25 15:30:00+08:00 2021-03-25 16:00:00+08:00
1     Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter           31 2021-03-25 15:30:00+08:00 2021-03-25 16:00:00+08:00
2   Tomcat-Requests_7441-"http-nio-8003"_RequestCo...           31 2021-03-25 15:30:00+08:00 2021-03-25 16:00:00+08:00
3   Tomcat-Requests_7441-"http-nio-8003"_Processin...           31 2021-03-25 15:30:00+08:00 2021-03-25 16:00:00+08:00
4   Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...           31 2021-03-25 15:30:00+08:00 2021-03-25 16:00:00+08:00
..                                                ...          ...                       ...                       ...
45                    OSLinux-CPU_CPU-2_SingleCpuidle           15 2021-03-25 15:31:00+08:00 2021-03-25 16:00:00+08:00
46     OSLinux-OSLinux_FILESYSTEM_----home_FSCapacity           15 2021-03-25 15:31:00+08:00 2021-03-25 16:00:00+08:00
47  OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSIn...           15 2021-03-25 15:31:00+08:00 2021-03-25 16:00:00+08:00
48  OSLinux-OSLinux_FILESYSTEM_----home_FSInodeUse...           15 2021-03-25 15:31:00+08:00 2021-03-25 16:00:00+08:00
49  OSLinux-OSLinux_FILESYSTEM_----boot_FSInodeUse...           15 2021-03-25 15:31:00+08:00 2021-03-25 16:00:00+08:00

[50 rows x 4 columns],     timestamp timestamp_dt_AsiaShanghai                                           kpi_name        value
0  1616657400 2021-03-25 15:30:00+08:00                         OSLinux-CPU_CPU_CPUCpuUtil    28.197000
1  1616657400 2021-03-25 15:30:00+08:00                            OSLinux-CPU_CPU_CPULoad     1.160000
2  1616657400 2021-03-25 15:30:00+08:00                         OSLinux-CPU_CPU_CPUSysTime     0.928161
3  1616657400 2021-03-25 15:30:00+08:00                        OSLinux-CPU_CPU_CPUUserTime    26.363107
4  1616657400 2021-03-25 15:30:00+08:00                             OSLinux-CPU_CPU_CPUWio     0.441189
5  1616657400 2021-03-25 15:30:00+08:00                        OSLinux-CPU_CPU_CPUidleutil    71.672355
6  1616657400 2021-03-25 15:30:00+08:00             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem  3779.000000
7  1616657400 2021-03-25 15:30:00+08:00           OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem   118.000000
8  1616657400 2021-03-25 15:30:00+08:00  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     0.300000
9  1616657400 2021-03-25 15:30:00+08:00          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem    58.000000,      timestamp timestamp_dt_AsiaShanghai          log_name   value
0   1616657400 2021-03-25 15:30:00+08:00   log.error_count    0.00
1   1616657400 2021-03-25 15:30:00+08:00  log.gc_pause_max    0.08
2   1616657400 2021-03-25 15:30:00+08:00     log.row_count  558.00
3   1616657460 2021-03-25 15:31:00+08:00   log.error_count    0.00
4   1616657460 2021-03-25 15:31:00+08:00  log.gc_pause_max    0.03
..         ...                       ...               ...     ...
45  1616658300 2021-03-25 15:45:00+08:00     log.row_count  486.00
46  1616658360 2021-03-25 15:46:00+08:00   log.error_count    0.00
47  1616658360 2021-03-25 15:46:00+08:00  log.gc_pause_max    0.11
48  1616658360 2021-03-25 15:46:00+08:00     log.row_count  495.00
49  1616658420 2021-03-25 15:47:00+08:00   log.error_count    0.00

[50 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing kernel variables if available
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure Tomcat01 full metric rows exist
tomcat_metric_rows_full = metric_df[metric_df['cmdb_id'] == 'Tomcat01'].copy()
tomcat_metric_rows_full['timestamp_dt_AsiaShanghai'] = pd.to_datetime(
    tomcat_metric_rows_full['timestamp'], unit='s', utc=True
).dt.tz_convert(tz)

# 1) Compute global P95 per kpi_name across full Tomcat01 series (before window filtering)
agg_tomcat_global = (
    tomcat_metric_rows_full
    .groupby('kpi_name', as_index=False)
    .agg(
        p95_value = ('value', lambda s: float(s.quantile(0.95))),
        total_count = ('value', 'size'),
        min_timestamp = ('timestamp', 'min'),
        max_timestamp = ('timestamp', 'max')
    )
)
agg_tomcat_global['min_timestamp_dt'] = pd.to_datetime(agg_tomcat_global['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
agg_tomcat_global['max_timestamp_dt'] = pd.to_datetime(agg_tomcat_global['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Store aggregation in kernel variable `agg_tomcat_global`
agg_tomcat_global = agg_tomcat_global

# 2) Filter Tomcat01 metric rows to the incident window
window_start = pd.Timestamp('2021-03-25 15:30:00', tz=tz)
window_end   = pd.Timestamp('2021-03-25 16:00:00', tz=tz)

tomcat_window = tomcat_metric_rows_full[
    (tomcat_metric_rows_full['timestamp_dt_AsiaShanghai'] >= window_start) &
    (tomcat_metric_rows_full['timestamp_dt_AsiaShanghai'] <= window_end)
].copy()

# Keep full per-kpi window rows for follow-up
tomcat_window_rows_by_kpi = {k: g.copy() for k, g in tomcat_window.groupby('kpi_name')}

# 2) For each kpi_name compute requested aggregates
rows = []
for kpi, group in tomcat_window.groupby('kpi_name'):
    # global p95 from agg_tomcat_global
    p95_row = agg_tomcat_global[agg_tomcat_global['kpi_name'] == kpi]
    p95_value = float(p95_row['p95_value'].values[0]) if not p95_row.empty else float('nan')
    
    window_count = int(group.shape[0])
    # compute above mask aligned to group index
    if np.isnan(p95_value):
        above_mask = pd.Series(False, index=group.index)
    else:
        above_mask = group['value'] > p95_value
    
    window_count_above = int(above_mask.sum())
    
    # earliest timestamp above p95 in Asia/Shanghai
    if window_count_above > 0:
        earliest_ts = group.loc[above_mask, 'timestamp_dt_AsiaShanghai'].min()
        earliest_iso = earliest_ts.isoformat()
    else:
        earliest_iso = None
    
    # max value in window
    max_value_in_window = float(group['value'].max()) if not group.empty else float('nan')
    
    # max_over_p95_ratio handling p95==0
    if np.isnan(p95_value):
        max_over_ratio = float('nan')
    else:
        if p95_value == 0:
            max_over_ratio = np.inf if max_value_in_window > 0 else 0.0
        else:
            max_over_ratio = (max_value_in_window - p95_value) / p95_value
    
    # longest consecutive minutes above p95
    longest_run = 0
    if window_count_above > 0:
        # sort by timestamp seconds
        g_sorted = group.sort_values('timestamp').reset_index(drop=True)
        # realign mask to sorted dataframe
        above_sorted = (g_sorted['value'] > p95_value) if not np.isnan(p95_value) else pd.Series(False, index=g_sorted.index)
        above_idx = g_sorted[above_sorted].copy().reset_index(drop=True)
        cnt_above = above_idx.shape[0]
        if cnt_above == 1:
            longest_run = 1
        elif cnt_above > 1:
            # compute diffs in seconds between successive above timestamps
            diffs = above_idx['timestamp'].diff().fillna(0).astype(int)
            grp = (diffs != 60).cumsum()
            run_lengths = above_idx.groupby(grp).size()
            if run_lengths.empty:
                longest_run = 0
            else:
                longest_run = int(run_lengths.max())
        else:
            longest_run = 0
    else:
        longest_run = 0
    
    rows.append({
        'kpi_name': (kpi[:200]) if isinstance(kpi, str) else kpi,
        'p95_value': p95_value,
        'window_count': window_count,
        'window_count_above_p95': window_count_above,
        'earliest_timestamp_above_p95': earliest_iso,
        'max_value_in_window': max_value_in_window,
        'max_over_p95_ratio': max_over_ratio,
        'longest_consecutive_minutes_above_p95': longest_run
    })

tomcat_kpi_window_summary = pd.DataFrame(rows)

# 3) Sort and limit to top 20 rows
tomcat_kpi_window_summary_sorted = tomcat_kpi_window_summary.sort_values(
    by=['longest_consecutive_minutes_above_p95','max_over_p95_ratio'],
    ascending=[False, False]
).reset_index(drop=True).head(20)

# Keep full per-kpi window rows in `tomcat_window_rows_by_kpi` for follow-up (already assigned)

# Display compact result
tomcat_kpi_window_summary_sorted

```
Out[7]:
```
summary = (
    "Tomcat01 KPI window summary (2021-03-25 15:30–16:00 Asia/Shanghai):\n"
    "- Computed global P95 per Tomcat01 KPI and inspected the incident window.\n"
    "- Top affected KPIs by longest consecutive exceedance (2 minutes):\n"
    "  * OSLinux-CPU_CPU_CPUidleutil — p95≈73.1599, earliest exceedance 15:58, max≈73.2837, ratio≈0.00169\n"
    "  * Tomcat-Requests (RequestCount) — p95≈3.422536e6, earliest 15:59, max≈3.423193e6, ratio≈0.000192\n"
    "  * Tomcat-Requests (Processing) — p95≈2.217806e9, earliest 15:59, max≈2.218053e9, ratio≈0.000112\n"
    "  * OSLinux-OSLinux_ZABBIX_Host_Uptime — p95≈7.858114e6, earliest 15:59, max≈7.858203e6, ratio≈0.000011\n"
    "- Notable single-minute but larger relative spikes:\n"
    "  * OSLinux-CPU_CPU_CPUWio — p95≈0.146006, earliest exceedance 15:30, max≈0.441189, ratio≈2.02 ( > 2x )\n"
    "  * Local disk and network KPIs (e.g., sda DSKBps, sdb DSKPer..., NETKBTotalPerSec) show single-minute exceedances with ratios up to ~1.86 and ~0.26 respectively.\n"
    "  * Tomcat thread count (http-nio-8003 CurrentThreads) — p95=10, earliest 15:30, max=14, ratio≈0.333.\n"
    "- Overall pattern: a few KPIs had 2-minute consecutive exceedances but with very small absolute relative increases (many ratios near 1e-4 to 1e-3). A handful of KPIs show larger relative spikes (CPU WIO, certain disk I/O metrics, thread count) at specific minutes (notably 15:30 and ~15:35–15:44).\n"
    "- For follow-up: full global P95 table is stored in `agg_tomcat_global`; full per-kpi window rows are stored in `tomcat_window_rows_by_kpi` for detailed inspection.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name     p95_value  window_count  window_count_above_p95 earliest_timestamp_above_p95  max_value_in_window  max_over_p95_ratio  longest_consecutive_minutes_above_p95
0                         OSLinux-CPU_CPU_CPUidleutil  7.315990e+01            30                       2    2021-03-25T15:58:00+08:00         7.328368e+01            0.001692                                      2
1   Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  3.422536e+06            31                       2    2021-03-25T15:59:00+08:00         3.423193e+06            0.000192                                      2
2   Tomcat-Requests_7441-"http-nio-8003"_Processin...  2.217806e+09            31                       2    2021-03-25T15:59:00+08:00         2.218053e+09            0.000112                                      2
3                  OSLinux-OSLinux_ZABBIX_Host_Uptime  7.858114e+06            30                       2    2021-03-25T15:59:00+08:00         7.858203e+06            0.000011                                      2
4                              OSLinux-CPU_CPU_CPUWio  1.460063e-01            30                       2    2021-03-25T15:30:00+08:00         4.411890e-01            2.021712                                      1
5      OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  3.500000e-01            14                       1    2021-03-25T15:35:00+08:00         1.000000e+00            1.857143                                      1
6   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...  4.200000e-01            15                       1    2021-03-25T15:44:00+08:00         7.000000e-01            0.666667                                      1
7                     OSLinux-CPU_CPU-0_SingleCpuUtil  1.889856e+01             9                       1    2021-03-25T15:53:00+08:00         2.647600e+01            0.400953                                      1
8                     OSLinux-CPU_CPU-1_SingleCpuUtil  6.579106e+01            10                       1    2021-03-25T15:54:00+08:00         9.182590e+01            0.395720                                      1
9      OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps  5.900000e+00            15                       1    2021-03-25T15:42:00+08:00         8.000000e+00            0.355932                                      1
10  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.050000e+01            31                       2    2021-03-25T15:30:00+08:00         1.400000e+01            0.333333                                      1
11    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec  6.744786e+05            15                       1    2021-03-25T15:31:00+08:00         8.524200e+05            0.263821                                      1
12                    OSLinux-CPU_CPU-2_SingleCpuidle  7.727763e+01            15                       1    2021-03-25T15:54:00+08:00         9.668995e+01            0.251202                                      1
13                            OSLinux-CPU_CPU_CPULoad  1.175500e+00            30                       2    2021-03-25T15:44:00+08:00         1.390000e+00            0.182476                                      1
14    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps  8.166800e-01            15                       1    2021-03-25T15:31:00+08:00         9.333000e-01            0.142798                                      1
15    OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil  6.000000e-03             9                       1    2021-03-25T15:31:00+08:00         6.600000e-03            0.100000                                      1
16   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite  1.060001e+01            15                       1    2021-03-25T15:31:00+08:00         1.106670e+01            0.044027                                      1
17  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...  2.699000e+02            10                       1    2021-03-25T15:31:00+08:00         2.780000e+02            0.030011                                      1
18   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  2.631600e+02            15                       1    2021-03-25T15:31:00+08:00         2.705333e+02            0.028018                                      1
19                    OSLinux-CPU_CPU-3_SingleCpuUtil  5.492940e+00            10                       1    2021-03-25T16:00:00+08:00         5.615700e+00            0.022349                                      1```
```